In [ ]:
import os, shutil, zipfile
if not os.path.isdir("data"):
    try:
        from google.colab import files
        print("Hay chon tep Lab07_data.zip (hoac cac tep .csv cua bai)")
        up = files.upload()
    except ImportError:
        raise FileNotFoundError("Khong thay thu muc data/, hay dat no canh notebook")
    os.makedirs("data", exist_ok=True)
    for ten in up:
        if ten.endswith(".zip"):
            zipfile.ZipFile(ten).extractall(".")
        elif ten.endswith(".csv"):
            shutil.move(ten, os.path.join("data", ten))
print(sorted(os.listdir("data")))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score,
                             roc_auc_score, roc_curve)
from sklearn.model_selection import cross_val_predict, cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

df = pd.read_csv("data/sinh_vien_bo_hoc.csv")
X = df.drop(columns="bo_hoc")
y = df["bo_hoc"]
# stratify=y giu dung ty le bo hoc o ca tap hoc lan tap kiem tra
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=17, stratify=y)
lr = make_pipeline(StandardScaler(), LogisticRegression())

In [ ]:
print("Kich thuoc bang:", df.shape)
print(df.head())
print()
print("So sinh vien moi lop:")
print(df["bo_hoc"].value_counts().to_string())
print(f"Ty le bo hoc: {df['bo_hoc'].mean():.3f}")
print()
print(f"Tap kiem tra: {len(y_te)} ban, {y_te.sum()} ban bo hoc")
print()

luoi = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
du_luoi = luoi.predict(X_te)
du_lr = lr.fit(X_tr, y_tr).predict(X_te)

for ten, du in [("Mo hinh luoi", du_luoi), ("Hoi quy logistic", du_lr)]:
    acc = accuracy_score(y_te, du)
    tim_ra = ((du == 1) & (y_te == 1)).sum()
    print(f"{ten:17s} accuracy {acc:.4f}, tim ra {tim_ra} tren {y_te.sum()} ban bo hoc")

In [ ]:
du = lr.fit(X_tr, y_tr).predict(X_te)
that = y_te.to_numpy()

tp = np.sum((that == 1) & (du == 1))
fp = np.sum((that == 0) & (du == 1))
fn = np.sum((that == 1) & (du == 0))
tn = np.sum((that == 0) & (du == 0))
print("Tu dem:")
print(f"  TP = {tp} (doan bo hoc, dung la bo hoc)")
print(f"  FP = {fp} (bao dong nham)")
print(f"  FN = {fn} (bo sot)")
print(f"  TN = {tn} (doan khong bo, dung la khong bo)")
print(f"  Tong = {tp + fp + fn + tn}")
print()

m = confusion_matrix(that, du)
print("confusion_matrix cua scikit-learn:")
print(m)
print("Thu tu khi trai ra: TN, FP, FN, TP =", m.ravel())
print()
acc = (tp + tn) / (tp + fp + fn + tn)
print(f"Accuracy tinh tu bon o: ({tp} + {tn}) / {len(that)} = {acc:.4f}")
print(f"So ban bi bao co nguy co: TP + FP = {tp + fp}")

In [ ]:
# Ve ma tran nham lan cua mo hinh luoi va cua logistic
fig, axs = plt.subplots(1, 2, figsize=(8, 3.5))
for ax, (ten, d) in zip(axs, [("Luon doan khong bo hoc", du_luoi), ("Logistic, nguong 0.5", du)]):
    mm = confusion_matrix(that, d)
    ax.imshow(mm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, mm[i, j], ha="center", va="center", fontsize=14)
    ax.set_xticks([0, 1], ["Khong bo", "Bo hoc"]); ax.set_yticks([0, 1], ["Khong bo", "Bo hoc"])
    ax.set_xlabel("Mo hinh doan"); ax.set_ylabel("Thuc te"); ax.set_title(ten)
plt.tight_layout(); plt.show()

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_te, du).ravel()
p = tp / (tp + fp)
r = tp / (tp + fn)
f1 = 2 * p * r / (p + r)
print("Tinh tay tu bon o:")
print(f"  Precision = {tp} / ({tp} + {fp}) = {p:.4f}")
print(f"  Recall    = {tp} / ({tp} + {fn}) = {r:.4f}")
print(f"  F1        = {f1:.4f}")
print()
print("Tinh bang scikit-learn:")
print(f"  Precision = {precision_score(y_te, du):.4f}")
print(f"  Recall    = {recall_score(y_te, du):.4f}")
print(f"  F1        = {f1_score(y_te, du):.4f}")
print()
print(classification_report(y_te, du, digits=4, target_names=["khong bo", "bo hoc"]))

In [ ]:
lr.fit(X_tr, y_tr)
xs = lr.predict_proba(X_te)[:, 1]   # xac suat bo hoc cua tung ban

print("Tap kiem tra, doi nguong")
print("nguong so bao  TP  FP  FN  precision recall    F1")
for t in [0.5, 0.4, 0.3, 0.2, 0.1]:
    d = (xs >= t).astype(int)
    tp_ = int(((d == 1) & (y_te == 1)).sum())
    fp_ = int(((d == 1) & (y_te == 0)).sum())
    fn_ = int(((d == 0) & (y_te == 1)).sum())
    print(f"  {t:.1f} {d.sum():5d} {tp_:4d}{fp_:4d}{fn_:4d}"
          f"   {precision_score(y_te, d):.4f}  {recall_score(y_te, d):.4f} {f1_score(y_te, d):.4f}")
print()

# Chon nguong dung cach: chi dung tap hoc, qua kiem dinh cheo
xs_hoc = cross_val_predict(lr, X_tr, y_tr, cv=5, method="predict_proba")[:, 1]
cac_nguong = np.round(np.arange(0.05, 0.96, 0.05), 2)
f1_hoc = [f1_score(y_tr, (xs_hoc >= t).astype(int)) for t in cac_nguong]
t_tot = cac_nguong[int(np.argmax(f1_hoc))]
print(f"Nguong cho F1 cao nhat tren tap hoc: {t_tot} (F1 kiem dinh cheo {max(f1_hoc):.4f})")

d = (xs >= t_tot).astype(int)
print(f"Tap kiem tra voi nguong {t_tot}: precision {precision_score(y_te, d):.4f},"
      f" recall {recall_score(y_te, d):.4f}, F1 {f1_score(y_te, d):.4f}")

In [ ]:
# Hinh: precision, recall, F1 theo nguong (tap kiem tra)
ng = np.arange(0.02, 0.91, 0.01)
P = [precision_score(y_te, (xs >= t).astype(int), zero_division=0) for t in ng]
R = [recall_score(y_te, (xs >= t).astype(int)) for t in ng]
F = [f1_score(y_te, (xs >= t).astype(int)) for t in ng]
plt.figure(figsize=(6, 3.5))
plt.plot(ng, P, label="Precision"); plt.plot(ng, R, "--", label="Recall"); plt.plot(ng, F, ":", label="F1")
plt.xlabel("Nguong xac suat"); plt.ylabel("Gia tri"); plt.legend(); plt.grid(alpha=.3)
plt.title("Ha nguong: tim ra nhieu hon, bao nham nhieu hon"); plt.show()

In [ ]:
xs = lr.fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
fpr, tpr, nguong = roc_curve(y_te, xs)
print("Duong ROC co", len(fpr), "diem. Vai diem dau:")
for a, b in list(zip(fpr, tpr))[:6]:
    print(f"  bao dong nham {a:.4f}  recall {b:.4f}")
print(f"AUC cua logistic = {roc_auc_score(y_te, xs):.4f}")
print()

mo_hinh = {
    "Logistic": lr,
    "Naive Bayes": GaussianNB(),
    "KNN, k = 15": make_pipeline(StandardScaler(), KNeighborsClassifier(15)),
    "Cay sau 4": DecisionTreeClassifier(max_depth=4, random_state=0),
    "Rung 200 cay": RandomForestClassifier(200, random_state=0),
    "SVM RBF": make_pipeline(StandardScaler(), SVC(probability=True, random_state=0)),
}
print("Mo hinh          AUC   recall  F1 (nguong 0.5)")
for ten, mh in mo_hinh.items():
    mh.fit(X_tr, y_tr)
    pp = mh.predict_proba(X_te)[:, 1]
    d = (pp >= 0.5).astype(int)
    print(f"  {ten:14s} {roc_auc_score(y_te, pp):.4f} {recall_score(y_te, d):.4f} {f1_score(y_te, d):.4f}")

In [ ]:
plt.figure(figsize=(4.5, 4.5))
plt.plot(fpr, tpr, label=f"Logistic, AUC = {roc_auc_score(y_te, xs):.2f}")
plt.plot([0, 1], [0, 1], "--", c="gray", label="Doan ngau nhien, AUC = 0.5")
for t, mau in [(0.5, "red"), (0.2, "green")]:
    k = np.argmin(np.abs(nguong - t))
    plt.scatter(fpr[k], tpr[k], c=mau, zorder=3, label=f"Nguong {t}")
plt.xlabel("Ty le bao dong nham"); plt.ylabel("Recall"); plt.legend(fontsize=8); plt.title("Duong ROC"); plt.show()

In [ ]:
# Chia khong phan tang: so ban bo hoc trong tap kiem tra dao dong
dem = []
for rs in range(10):
    _, _, _, y_te_thu = train_test_split(X, y, test_size=0.25, random_state=rs)
    dem.append(int(y_te_thu.sum()))
print("Khong stratify, so ban bo hoc trong tap kiem tra:", dem)
print()

print("Kiem dinh cheo 5 phan tren tap hoc:")
for tm in ["accuracy", "precision", "recall", "f1"]:
    d = cross_val_score(lr, X_tr, y_tr, cv=5, scoring=tm)
    print(f"  {tm:10s} {d.mean():.4f}")
print()

cb = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced"))
print("Tap kiem tra     precision recall    F1")
for ten, mh in [("Binh thuong", lr), ("Can bang lop", cb)]:
    d = mh.fit(X_tr, y_tr).predict(X_te)
    print(f"  {ten:12s} {precision_score(y_te, d):.4f}    {recall_score(y_te, d):.4f} {f1_score(y_te, d):.4f}")

---
# Bài tập


In [ ]:
def danh_gia(y_that, y_doan):
    y_that = np.asarray(y_that); y_doan = np.asarray(y_doan)
    tp = np.sum((y_that == 1) & (y_doan == 1))
    fp = np.sum((y_that == 0) & (y_doan == 1))
    fn = np.sum((y_that == 1) & (y_doan == 0))
    tn = np.sum((y_that == 0) & (y_doan == 0))
    acc = (tp + tn) / len(y_that)
    # chia cho 0 -> coi bang 0 (giong zero_division=0 cua scikit-learn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return acc, precision, recall, f1

for ten, d in [("Logistic", du_lr), ("Mo hinh luoi", du_luoi)]:
    print(ten)
    print("  Tu viet     :", np.round(danh_gia(y_te, d), 4))
    print("  scikit-learn:", np.round([accuracy_score(y_te, d),
          precision_score(y_te, d, zero_division=0),
          recall_score(y_te, d), f1_score(y_te, d, zero_division=0)], 4))

In [ ]:
from sklearn.datasets import load_digits
Xd, yd = load_digits(return_X_y=True)
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(Xd, yd, test_size=0.25, random_state=17, stratify=yd)
svm_d = SVC().fit(Xd_tr, yd_tr)
du_d = svm_d.predict(Xd_te)
print(classification_report(yd_te, du_d, digits=4))
rep = pd.DataFrame(classification_report(yd_te, du_d, output_dict=True)).T.iloc[:10]
print("Recall thap nhat   : chu so", rep["recall"].idxmin(), round(rep["recall"].min(), 4))
print("Precision thap nhat: chu so", rep["precision"].idxmin(), round(rep["precision"].min(), 4))

In [ ]:
xs_hoc = cross_val_predict(lr, X_tr, y_tr, cv=5, method="predict_proba")[:, 1]
cac_nguong = np.round(np.arange(0.01, 0.99, 0.01), 2)
du_dieu_kien = [t for t in cac_nguong if recall_score(y_tr, (xs_hoc >= t).astype(int)) >= 0.8]
t_chon = max(du_dieu_kien)
print("Nguong lon nhat cho recall >= 0.8 tren tap hoc:", t_chon)

lr.fit(X_tr, y_tr)
d = (lr.predict_proba(X_te)[:, 1] >= t_chon).astype(int)
print(f"Tap kiem tra: recall {recall_score(y_te, d):.4f}, precision {precision_score(y_te, d):.4f},"
      f" so ban bi bao {d.sum()}")

## Bài tập 4: Tự vẽ đường ROC


In [ ]:
rf = RandomForestClassifier(200, random_state=0).fit(X_tr, y_tr)
p_lr = lr.fit(X_tr, y_tr).predict_proba(X_te)[:, 1]
p_rf = rf.predict_proba(X_te)[:, 1]

plt.figure(figsize=(5, 5))
for ten, pp in [("Logistic", p_lr), ("Rung ngau nhien", p_rf)]:
    f_, t_, _ = roc_curve(y_te, pp)
    plt.plot(f_, t_, label=f"{ten}, AUC = {roc_auc_score(y_te, pp):.4f}")
plt.plot([0, 1], [0, 1], "--", c="gray", label="Doan ngau nhien")
plt.xlabel("Ty le bao dong nham"); plt.ylabel("Recall"); plt.legend(); plt.grid(alpha=.3)
plt.savefig("roc.png", dpi=150, bbox_inches="tight"); plt.show()

## Bài tập 5: Cân bằng lớp cho mô hình khác


In [ ]:
cap = {
    "Rung ngau nhien": (RandomForestClassifier(200, random_state=0),
                        RandomForestClassifier(200, random_state=0, class_weight="balanced")),
    "SVM RBF": (make_pipeline(StandardScaler(), SVC()),
                make_pipeline(StandardScaler(), SVC(class_weight="balanced"))),
}
dong = []
for ten, (thuong, cb_) in cap.items():
    for nhan, mh in [("binh thuong", thuong), ("can bang", cb_)]:
        d = mh.fit(X_tr, y_tr).predict(X_te)
        dong.append({"mo hinh": ten, "cach": nhan,
                     "precision": round(precision_score(y_te, d, zero_division=0), 4),
                     "recall": round(recall_score(y_te, d), 4),
                     "F1": round(f1_score(y_te, d), 4)})
pd.DataFrame(dong)

## Bài tập 6: MAE và RMSE với một lần đoán hỏng


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
nha = pd.read_csv("data/gia_nha.csv")
Xn = nha.drop(columns="gia"); yn = nha["gia"]
Xn_tr, Xn_te, yn_tr, yn_te = train_test_split(Xn, yn, test_size=0.25, random_state=17)
dd = LinearRegression().fit(Xn_tr, yn_tr).predict(Xn_te)

def bao_cao(ten, d):
    mae = mean_absolute_error(yn_te, d)
    rmse = np.sqrt(mean_squared_error(yn_te, d))
    print(f"{ten:18s} MAE {mae:.4f}  RMSE {rmse:.4f}  R2 {r2_score(yn_te, d):.4f}")

bao_cao("Binh thuong", dd)
dd_hong = dd.copy()
dd_hong[0] += 5          # mot can bi doan hong them 5 ty
bao_cao("Mot can lech +5", dd_hong)